In [1]:
%load_ext autoreload
%autoreload 2


In [ ]:
# Test 1: Basic Elasticsearch search
from explain.literature._esearch_utils import search_indexes

keywords = ["erlotinib", "EGFR"]

articles = await search_indexes(
    keywords=keywords, indexes="full", top_k=10, disease_id="D008107"
)

2025-08-25 12:07:26.365 | DEBUG    | explain.literature._esearch_utils:retrieve_article:252 - Retrieved 10 hits
2025-08-25 12:07:26.396 | DEBUG    | explain.literature._esearch_utils:retrieve_article:303 - Retrieved paper IDs: ['pubmed_32872164', 'pubmed_37026519', 'pubmed_33396222', 'pubmed_32532965', 'pubmed_30854137', 'pubmed_34944063', 'pubmed_38711992', 'pubmed_38240082', 'pubmed_28338617', 'pubmed_32531926']


[{'_index': 'full-text-articles', '_id': 'pubmed_32872164', '_score': 364.44864, '_source': {'genes': ['ENSEMBL:ENSG00000197461', 'ENSEMBL:ENSG00000105974', 'ENSEMBL:ENSG00000140009', 'ENSEMBL:ENSG00000171109', 'ENSEMBL:ENSG00000141510', 'ENSEMBL:ENSG00000105355', 'ENSEMBL:ENSG00000072310', 'ENSEMBL:ENSG00000087470', 'ENSEMBL:ENSG00000155792', 'ENSEMBL:ENSG00000134853', 'ENSEMBL:ENSG00000141736', 'ENSEMBL:ENSG00000144895', 'ENSEMBL:ENSG00000112561', 'ENSEMBL:ENSG00000169710', 'ENSEMBL:ENSG00000143799', 'ENSEMBL:ENSG00000114650', 'ENSEMBL:ENSG00000177169', 'ENSEMBL:ENSG00000172071', 'ENSEMBL:ENSG00000170323', 'ENSEMBL:ENSG00000132109', 'ENSEMBL:ENSG00000109819', 'ENSEMBL:ENSG00000132170', 'ENSEMBL:ENSG00000131791', 'ENSEMBL:ENSG00000138798', 'ENSEMBL:ENSG00000111640', 'ENSEMBL:ENSG00000197122', 'ENSEMBL:ENSG00000103510', 'ENSEMBL:ENSG00000057252', 'ENSEMBL:ENSG00000188612', 'ENSEMBL:ENSG00000174059', 'ENSEMBL:ENSG00000142208', 'ENSEMBL:ENSG00000012779', 'ENSEMBL:ENSG00000100219', 'ENSEM

In [ ]:
# Test 2: Evidence-based search with BigQuery vector search
from explain.literature.evidence_query import process_literature_query
from explain.llm import set_env_secrets 
set_env_secrets()

# Define a scientific question and keywords
query = "What is the relationship between EGFR mutations and imatinib resistance in cancer?"
keywords = ["EGFR", "imatinib", "resistance", "mutation"]

# Run the full evidence-based search pipeline
# This combines Elasticsearch + BigQuery vector search + LLM answer generation
response = await process_literature_query(
    query=query,
    keywords=keywords,
    top_k=20,  # Number of papers to retrieve from Elasticsearch
    vector_top_k=10,  # Number of semantic snippets from BigQuery
    embedding_model="text-embedding-3-small",
    llm_model="gpt-4o-mini",  # Using a faster model for testing
    generate_answer=True,
    evidence_only=True  # Answer strictly from evidence
)

# Display results
print(f"\\n=== Evidence-Based Search Results ===")
print(f"Elasticsearch papers found: {response.get('num_elasticsearch_results', 0)}")
print(f"Vector search snippets found: {response.get('num_vector_results', 0)}")

if response.get('generated_answer'):
    print(f"\\n=== Generated Answer ===")
    print(response['generated_answer'][:500] + "..." if len(response['generated_answer']) > 500 else response['generated_answer'])
    
if response.get('evidence_summary'):
    print(f"\\n=== Evidence Summary (first 3) ===")
    for i, evidence in enumerate(response['evidence_summary'][:3]):
        print(f"{i+1}. {evidence.get('text', '')[:150]}...")
        print(f"   Source: {evidence.get('source', 'Unknown')}")
        print()
else:
    print("\\nNo evidence found")

2025-08-25 12:09:22.014 | DEBUG    | explain.literature.evidence_query:process_literature_query:130 - Starting Elasticsearch search...
2025-08-25 12:09:23.438 | DEBUG    | explain.literature._esearch_utils:retrieve_article:252 - Retrieved 20 hits
2025-08-25 12:09:23.448 | DEBUG    | explain.literature._esearch_utils:retrieve_article:303 - Retrieved paper IDs: ['pubmed_24847446', 'pubmed_28469513', 'pubmed_36982592', 'pubmed_19204794', 'pubmed_27992367', 'pubmed_37223538', 'pubmed_23130349', 'pubmed_26316776', 'pubmed_27005669', 'pubmed_28031906', 'pubmed_31998490', 'pubmed_23904849', 'pubmed_39035746', 'pubmed_22211140', 'pubmed_21897799', 'pubmed_34899235', 'pubmed_17970609', 'pubmed_34987800', 'pubmed_16404421', 'pubmed_36838629']
2025-08-25 12:09:23.450 | DEBUG    | explain.literature.evidence_query:process_literature_query:138 - Elasticsearch returned 20 articles


[{'_index': 'full-text-articles', '_id': 'pubmed_24847446', '_score': 124.736374, '_source': {'genes': ['ENSEMBL:ENSG00000133101', 'ENSEMBL:ENSG00000140443', 'ENSEMBL:ENSG00000198793', 'ENSEMBL:ENSG00000157404', 'ENSEMBL:ENSG00000048052', 'ENSEMBL:ENSG00000197122', 'ENSEMBL:ENSG00000123374', 'ENSEMBL:ENSG00000198400', 'ENSEMBL:ENSG00000146648', 'ENSEMBL:ENSG00000111537'], 'diseases': ['http://id.nlm.nih.gov/mesh/D006973', 'http://id.nlm.nih.gov/mesh/D052016', 'http://id.nlm.nih.gov/mesh/D002277', 'http://id.nlm.nih.gov/mesh/D009369', 'http://id.nlm.nih.gov/mesh/D013945', 'http://id.nlm.nih.gov/mesh/D005221'], 'parent_diseases': ['http://id.nlm.nih.gov/mesh/D013953', 'http://id.nlm.nih.gov/mesh/D009059', 'http://id.nlm.nih.gov/mesh/D006973', 'http://id.nlm.nih.gov/mesh/D005759', 'http://id.nlm.nih.gov/mesh/D009370', 'http://id.nlm.nih.gov/mesh/D052016', 'http://id.nlm.nih.gov/mesh/D005767', 'http://id.nlm.nih.gov/mesh/D014652', 'http://id.nlm.nih.gov/mesh/D009375', 'http://id.nlm.nih.go

2025-08-25 12:09:24.980 | DEBUG    | explain.literature._bq_process_paper:check_existing_papers_in_bigquery:218 - Found 0 papers already in BigQuery
2025-08-25 12:09:24.981 | DEBUG    | explain.literature.evidence_query:process_literature_query:148 - Need to process 20 new papers
2025-08-25 12:09:25.023 | DEBUG    | explain.literature.evidence_query:process_literature_query:170 - Created 89 chunks from 20 papers in parallel
2025-08-25 12:09:25.024 | INFO     | explain.literature._bq_process_paper:aembed_chunks:98 - Running 2 embedding batches concurrently (avg_length: 3811 chars)...
2025-08-25 12:09:28.672 | INFO     | explain.literature._bq_process_paper:aembed_chunks:119 - Successfully embedded 89/89 chunks
2025-08-25 12:09:29.213 | DEBUG    | explain.literature._bq_process_paper:ensure_dataset_exists:285 - Dataset rxrx-medchem-auto-dev.medchem_auto_dev_bigquery_db already exists
2025-08-25 12:09:29.631 | DEBUG    | explain.literature._bq_process_paper:ensure_table_exists:317 - Table

\n=== Evidence-Based Search Results ===
Elasticsearch papers found: 20
Vector search snippets found: 10
Total evidence pieces: 0
\n=== Generated Answer ===
The relationship between EGFR mutations and imatinib resistance in cancer appears to be multifaceted, particularly evident in studies involving different cancer types. Evidence indicates that specific mutations, notably in the EGFR and c-KIT genes, can lead to resistance against targeted therapies like imatinib. For example, in gastrointestinal stromal tumors (GISTs), mutations in exon 17 of the c-KIT gene, such as N822K, have been associated with resistance to imatinib and subsequent therapies,...
\nNo evidence found
